In [ ]:
import pandas as pd
import datetime as dt
import numpy as np
import matplotlib.pyplot as plt

## Total Returns Extended to Commodities

### Data sources
- st louis 3m tbill used as risk free rate for determination of sharpe, it can be used as approximation funding cost available since 1930
- shiller 10y bond rate and equity return and div, this allows computing 10y total return and equity total return
- spot commo prices from CMO, spot can be used approx for return on precious metals, and possibly copper. Others, notable oil spot price return is missing massive future roll return
- CL1 and CL6 historical data from bloomberg, since 1991. This allows computing future roll return, which is significant for oil.

In [ ]:
def tbillrate():
    df = pd.read_csv("data/TB3MS.csv")
    df["observation_date"] = pd.to_datetime(df["observation_date"])
    df["TB3MS"] /= 100
    return df.set_index("observation_date")
def compute_gold_tr(df):
    for colname in df.columns:
        df[colname+'TR'] = df[colname]/df[colname].shift(1)
    return df
def goldprice(col):
    df = pd.read_csv("data/cmo-data-monthly.csv")
    print(df.columns)
    df = df[["date"]+list(col.keys())]
    df["date"] = pd.to_datetime(df["date"])+dt.timedelta(days=1)
    df = df.loc[df["date"]>=dt.datetime(1971,2,1)]
    return compute_gold_tr(df.set_index("date").rename(columns=col))
def read_shiller_out(col):
    df = pd.read_csv("data/shiller_out.csv")
    df['Date'] = pd.to_datetime(df['Date'], format="%Y-%m-%d")+dt.timedelta(days=-14) # imported as 14th, will be used as 1st 
    df = df.set_index("Date")
    #for c in df.columns:
    #    df[c] = df[c].astype(float)
    #df['Rate10y'] = df['Rate10y']/100
    return df.join(tbillrate()).join(goldprice(col),how="inner")

"""
def compute_bond_tr(df):
    r = df['Rate10y']
    T = 10
    duration = -(1-np.exp(-r*T))/r
    bondcarry = r.shift(1)/12
    bondtotalret = 1+(r-r.shift(1))*duration+bondcarry
    df["bondTR"] = bondtotalret
def compute_eq_tr(df):
    df['eqTR'] = df['SP500']/df['SP500'].shift(1)+df['Div']/12/df['SP500']
def compute_cpi_tr(df):
    df['cpiTR'] = 1+df['CPI'].pct_change()
    """
def compute_tbill_tr(df):
    df['tbTR'] = 1+df['TB3MS'].shift(1)*365/360/12 # compute with shift with last rate, as if this was a 1m rate

def metrics_monthly_ret(df):
    trcolumns = [c for c in df.columns if "TR"==c[-2:] and c!="cpiTR"]
    logret = np.log(df[trcolumns].dropna())
    mu,sigma = logret.mean()*12,logret.std()*np.sqrt(12)
    sigma["tbTR"] = 0
    mu += 0.5*sigma**2
    r = mu["tbTR"]
    riskycol = [c for c in trcolumns if c != 'tbTR']
    print(riskycol)
    mustar = np.expm1(mu[riskycol]-r)
    data = {'mustar':mustar,'sigma':sigma[riskycol]}
    data['sharpe'] = data['mustar']/data['sigma']
    corr = logret[riskycol].corr()
    cov = logret[riskycol].cov()*12
    wstar = np.linalg.solve(cov,mustar)
    K = np.sum(wstar)
    data['w'] = wstar/K
    for c in riskycol:
        data[f'rho({c[:-2]})'] = corr[c]
    return pd.DataFrame(data,index=riskycol),r,K

def show_returns(df,filename,excl):
    cols = [c for c in df.columns if c[-2:]=="TR" and c not in excl]
    df = df[cols]
    dfmetrics,r,K = metrics_monthly_ret(df)
    print(dfmetrics.index)
    for c in dfmetrics.index:
        plt.plot(np.cumprod(df[c]),
                label=f"{c[:-2]}: $\mu^*$={dfmetrics.loc[c,'mustar']:.1%}, $\sigma$={dfmetrics.loc[c,'sigma']:.0%}, S={dfmetrics.loc[c,'sharpe']:.2f}, w={dfmetrics.loc[c,'w']:.0%}")
    plt.legend()
    plt.title(f"Asset Total Return r={r:.1%} K={K:.1f}")
    plt.ylabel("log total return")
    plt.yscale('log')
    datesstr = f"from {str(df.index[0])[:10]} to {str(df.index[-1])[:10]}"
    plt.xlabel(datesstr)
    print(datesstr)
    print(f"r={r:.2%} K={K:.1f} (kelly leverage)")
    plt.grid(True)  
    if not filename is None:
        plt.savefig(filename)
        plt.close()
    else:
        plt.show()
    return dfmetrics

def show_col_returns(col,show,excl):
    df = read_shiller_out(col)
    compute_tbill_tr(df)
    assets = [c[:-2] for c in df.columns if "TR" in c and c not in ["cpiTR","tbTR"]]
    filename = None if show else "_".join(assets).replace(" (no roll)","noroll")+".png"
    dfmetrics = show_returns(df.loc[df.index>=dt.datetime(1960,1,1)],filename,excl)
    return dfmetrics,filename


### Method
- tbill total return $\mu_r= 1+r . 365/360/12$ where $r$ is the rate in ACT.360 convention
- equity total return $\mu_e=P(i+1)/P(i) + D(i)/12$, where $P$ is price, $D$ annual dividend
- bond total return $\mu_b=A(i+1) (y(i+1)-y(i)) + y(i) / 12$ where $A$ is the annuity, $y$ the 10y yield
- commo price return $\mu_o=S(i+1)/S(i)$ where $S$ is spot price (this is total return for precious metals)
- contango amount $c=(F(i+6)-F(i+1))/F(i+6)$, contango rate $y_c=c^{1/5}$
- commo future total return $\mu_o=S(i+1)/S(i)-y_c$ 

### CMO Data
- crude oil spot shows big supply shocks in the 70s, return is significant
- gold, platinum, and silver are similar but gold has the lowest vol, 
- copper only recently started to beat inflation  from the 2000s (energiewende or China demand growing)
- softs tend to grow only 2%-3%, so return is flat above inflation, no excess return.
- algo prefers gold and copper to silver and platinum. The latter seems to be within efficient frontier of gold and copper.

### Metrics and Optimal Portfolio
- monthly log ret annualized vol $\sigma$
- monthly annualized expected return $\mu$ 
- monthly annualized expected excess return $\mu^*=\mu-r$
- Sharpe = $\mu^*/\sigma$
- optimal weightws $w^* = \Sigma^{-1} \mu^*$
- Optimal Kelly leverage $K=\sum(w)$
- normed weights $w = w^*/K$

In [ ]:
col = {"CRUDE_DUBAI":"crude (no roll)", "GOLD":"gold"}
dfmetrics,filename = show_col_returns(col,show=True,excl=[])
print(dfmetrics.to_markdown(floatfmt=".2%"))


In [ ]:
col = {"CRUDE_DUBAI":"crude (no roll)", "GOLD":"gold","COPPER":"copper"}
dfmetrics,filename = show_col_returns(col,show=True,excl=[])
print(dfmetrics.to_markdown(floatfmt=".2%"))


In [ ]:
dfcl = pd.read_csv("data/cl.csv")
dfcl["date"] = pd.to_datetime(dfcl["date"])
dfcl["oilCarryStratTR"] = pd.NA
dfcl["oilTR"] = pd.NA
dfcl = dfcl.set_index("date")
dfcl = dfcl.resample('M').last()
dfcl = dfcl.reset_index()
dfcl["date"] = dfcl["date"]+dt.timedelta(days=1)
dfcl = dfcl.set_index("date")
df = read_shiller_out({'GOLD': 'gold', 'COPPER': 'copper'})
dfcl = dfcl.join(df)
compute_tbill_tr(dfcl)
contango = (1+(dfcl["CL6"]-dfcl["CL1"])/dfcl["CL6"])**(1/5)-1-dfcl["TB3MS"]/12 # net carry
totret   = dfcl["CL6"].pct_change()-contango.shift(1)+1#+(dfcl["tbTR"]-1) # excess return mu*_o=mu_o-mu_
dfcl["oilCarryStratTR"] = 1+(totret-1)*np.where(contango<0.0088,1,-1) 
dfcl["oilTR"] = totret
dfcl["cumbackwardationpnl"] = np.cumprod(dfcl["oilCarryStratTR"])
dfcl["contango"] = contango
dfcl

## Single Asset Strategies to outperform Long Only

### Oil Contango based Strategy
- most time is backwardation
- backwardation appears bullish, contango is bearish
- oil total return shows future total return, oil carry strategy goes long in backwardation, short in contango

In [ ]:
x = (dfcl["contango"]).shift(1)
y = totret-1
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
errstd = np.std(y_clean - y_pred)
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.axvline(x=-b/a,color="black")
plt.xlabel("oil contango")
plt.ylabel("WTI 6M oil future total return")
plt.title(f"Oil total return vs contango - pivot: {-b/a:.2%}")
plt.show()
print(a,b,errstd)

In [ ]:
plt.plot(dfcl["cumbackwardationpnl"],label="oil carry strat")
# plt.plot(dfcl.loc[dfcl["backwardTR"]==1].index, 
#          dfcl.loc[dfcl["backwardTR"]==1, "cumbackwardationpnl"], 
#          color="red", 
#          linestyle='none',     # <-- this removes the line
#          marker='o',           # circle marker
#          markersize=1,
#          label="contango")
plt.plot(np.cumprod(dfcl["oilTR"]),label="long oil future total return")
plt.yscale('log')
plt.legend()
plt.grid(True)

In [ ]:
dfcl[[c for c in dfcl.columns if c[-2:]=="TR"]]

In [ ]:
metrics,r,K = metrics_monthly_ret(dfcl)
print(r,K)
print(metrics.to_markdown(floatfmt=".2%"))

## Future roll impact on pnl for oil
- data from 1991 only
- impact of backwardation is massively positive
- be careful with oil front contract, it gets negative in mar 2020
- we can use oil future total return, or much better, oil carry strategy

In [ ]:
print(show_returns(dfcl,filename=None,excl=["oilCarryStratTR"]).to_markdown(floatfmt=".2%"))

In [ ]:
print(show_returns(dfcl,filename=None,excl=["oilTR"]).to_markdown(floatfmt=".2%"))

### Rate study
- 10y bond is considered risky even though gov can print money because it bears rate risk
- 3m bond is considered risk free
- 10y vs 3m scatter plot shows correlation
- bond net return is flat vs 3m rate
- bond net return has slop vs 10y03m spread

In [ ]:
x = dfcl["TB3MS"]
y = dfcl["Rate10y"]
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
errstd = np.std(y_clean - y_pred)
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.xlabel("3M tbill rate")
plt.ylabel("10y rate")
plt.title("10y rate vs 3m rate")
plt.show()
x = (dfcl["TB3MS"]).shift(1)
y = dfcl["bondTR"]-dfcl["tbTR"]
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
errstd = np.std(y_clean - y_pred)
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.xlabel("3m rate")
plt.ylabel("bond net total return")
plt.title(f"bond net return vs 3m rate - pivot:{-b/a:.2%}")
plt.show()
print(a,b,errstd)
x = (dfcl["Rate10y"]-dfcl["TB3MS"]).shift(1)
y = dfcl["bondTR"]-dfcl["tbTR"]
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
errstd = np.std(y_clean - y_pred)
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.axvline(x=-b/a,color="black")
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.xlabel("10y - 3m rate spread")
plt.ylabel("bond net total return")
plt.title(f"bond net return vs rate spread - pivot: {-b/a:.2%}")
plt.show()
print(a,b,errstd)

In [ ]:
print(dfcl.columns)
bondcarrystrat = 1+(dfcl["bondTR"]-dfcl["tbTR"])*np.where((dfcl["Rate10y"]-dfcl["TB3MS"]).shift(1)>-0.008,1,-1)
plt.plot(np.cumprod(bondcarrystrat),label="bond carry strat")
plt.plot(np.cumprod(1+(dfcl["bondTR"]-dfcl["tbTR"])),label="bond long")
plt.legend()
plt.yscale('log')
plt.grid(True)
plt.show()


## Gold return
- gold does well when inflation fear is high: when bonds don't return much over tbills.

In [ ]:
x = (dfcl["goldTR"]-dfcl["tbTR"]).rolling(3).mean().shift(1)
y = dfcl["goldTR"]-dfcl["tbTR"]
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
errstd = np.std(y_clean - y_pred)
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.axvline(x=-b/a,color="black")
plt.xlabel("10y-3m spread")
plt.ylabel("gold net total return")
plt.title(f"Gold net return vs 10y-3m spread - pivot: {-b/a:.2%}")
plt.show()
print(a,b,errstd)

In [ ]:
goldcarrystrat = 1+(dfcl["goldTR"]-dfcl["tbTR"])*np.where(x>0,1,-1)
plt.plot(np.cumprod(goldcarrystrat),label="gold carry strat")
plt.plot(np.cumprod(1+(dfcl["goldTR"]-dfcl["tbTR"])),label="gold long")
plt.legend()
plt.yscale('log')
plt.grid(True)
plt.show()


## Copper has econ PhD
- copper move are highly correlated to past 3m equity return

In [ ]:
x = (dfcl["eqTR"]-dfcl["tbTR"]).rolling(3).mean().shift(1)
y = dfcl["copperTR"]-dfcl["tbTR"]
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
errstd = np.std(y_clean - y_pred)
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.axvline(x=-b/a,color="black")
plt.xlabel("past 3m mean eq net return")
plt.ylabel("copper excess return")
plt.title(f"copper excess retun vs past eq excess return pivot: {-b/a:.2%}")
plt.show()
print(a,b,errstd)

## Eq future return
- appears to be strongly correlated to prev 8m perf

In [ ]:
x = (dfcl["eqTR"]-dfcl["tbTR"]).rolling(8).mean().shift(1)
y = dfcl["eqTR"]-dfcl["tbTR"]
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
errstd = np.std(y_clean - y_pred)
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.axvline(x=-b/a,color="black")
plt.xlabel("past 12m mean eq net return")
plt.ylabel("eq return")
plt.title(f"eq retun vs past eq return pivot: {-b/a:.2%}")
plt.show()
print(a,b,errstd)